In [1]:
import torch

In [2]:
from thop import profile

In [4]:
'''
Implementation of ViTSTR based on timm VisionTransformer.

TODO: 
1) distilled deit backbone
2) base deit backbone

Copyright 2021 Rowel Atienza
'''

from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
from modules.resnet18 import ResNet18
import torch 
import torch.nn as nn
import torch.nn.functional as F
import logging
import torch.utils.model_zoo as model_zoo
#from torch.nn import LayerNorm
from copy import deepcopy
from functools import partial
from timm.models.vision_transformer import VisionTransformer, _cfg
from timm.models.registry import register_model
from timm.models import create_model
from timm.models.vision_transformer import Mlp, DropPath
import numpy as np
_logger = logging.getLogger(__name__)

__all__ = [
    'vitstr_tiny_patch16_224', 
    'vitstr_small_patch16_224', 
    'vitstr_base_patch16_224','vitstr_mae_custom',
    #'vitstr_tiny_distilled_patch16_224', 
    #'vitstr_small_distilled_patch16_224',
    #'vitstr_base_distilled_patch16_224',
]

class LayerNorm(nn.Module):

    def forward(self, x):
            # print(f"LayerNorm forward Input device: {x.device}")  # Debug
            # print(f"LayerNorm forward Model device: {next(self.parameters()).device}")
            return F.layer_norm(x, x.size()[1:], weight=None, bias=None, eps=1e-05)

def create_vitstr(num_tokens, model=None, checkpoint_path='', **kwargs):
    # Special handling for the MAE custom model
    if model == 'vitstr_mae_custom':
        vitstr = create_model(
            model,
            pretrained=False,  # No pretrained weights available by default
            nb_cls=num_tokens,
            checkpoint_path=checkpoint_path,
            **kwargs)
    else:
        vitstr = create_model(
            model,
            pretrained=True,
            num_classes=num_tokens,
            checkpoint_path=checkpoint_path,
            **kwargs)

    # Reset classifier to ensure proper initialization
    vitstr.reset_classifier(num_classes=num_tokens)

    return vitstr

class Block(nn.Module):
    def __init__(
            self,
            dim,
            num_heads,
            num_patches,
            mlp_ratio=4.,
            qkv_bias=False,
            drop=0.0,
            attn_drop=0.,
            init_values=None,
            drop_path=0.,
            act_layer=nn.GELU,
            norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_patches, num_heads=num_heads, qkv_bias=qkv_bias, 
                            attn_drop=attn_drop, proj_drop=drop)
        self.ls1 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.drop_path1 = DropPath(drop_path) if drop_path > 0. else nn.Identity()

        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), 
                      act_layer=act_layer, drop=drop)
        self.ls2 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.drop_path2 = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        # print(f"Block forward Input device: {x.device}")  # Debug
        # print(f"Block forward Model device: {next(self.parameters()).device}")
        x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x))))
        x = x + self.drop_path2(self.ls2(self.mlp(self.norm2(x))))
        return x
    
class Attention(nn.Module):
    def __init__(self, dim, num_patches, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        assert dim % num_heads == 0, 'dim should be divisible by num_heads'
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.num_patches = num_patches
        
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        # print(f"Class Attention forward Input device: {x.device}")  # Debug
        # print(f"Class Attention forward Model device: {next(self.parameters()).device}")

        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class LayerScale(nn.Module):
    def __init__(self, dim, init_values=1e-5, inplace=False):
        super().__init__()
        self.inplace = inplace
        self.gamma = nn.Parameter(init_values * torch.ones(dim))

    def forward(self, x):
        # print(f"LayerScale forward Input device: {x.device}")  # Debug
        # print(f"LayerScale forward Model device: {next(self.parameters()).device}")
        return x.mul_(self.gamma) if self.inplace else x * self.gamma

def get_2d_sincos_pos_embed(embed_dim, grid_size):
    """
    grid_size: (int, int) of the grid height and width
    return:
    pos_embed: [grid_size[0]*grid_size[1], embed_dim]
    """
    grid_h = np.arange(grid_size[0], dtype=np.float32)
    grid_w = np.arange(grid_size[1], dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)  # here w goes first
    grid = np.stack(grid, axis=0)

    grid = grid.reshape([2, 1, grid_size[0], grid_size[1]])
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    return pos_embed

def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0

    # use half of dimensions to encode grid_h
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])  # (H*W, D/2)
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])  # (H*W, D/2)

    emb = np.concatenate([emb_h, emb_w], axis=1)  # (H*W, D)
    return emb

def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    """
    embed_dim: output dimension for each position
    pos: a list of positions to be encoded: size (M,)
    out: (M, D)
    """
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=np.float64)
    omega /= embed_dim / 2.
    omega = 1. / 10000**omega  # (D/2,)

    pos = pos.reshape(-1)  # (M,)
    out = np.einsum('m,d->md', pos, omega)  # (M, D/2), outer product

    emb_sin = np.sin(out)  # (M, D/2)
    emb_cos = np.cos(out)  # (M, D/2)

    emb = np.concatenate([emb_sin, emb_cos], axis=1)  # (M, D)
    return emb






# class MaskedAutoencoderViT(nn.Module):
#     def __init__(self, nb_cls, img_size=(224, 224), patch_size=(16, 16), embed_dim=768,
#                  depth=4, num_heads=6, mlp_ratio=4., norm_layer=nn.LayerNorm,
#                  mask_ratio=0.75, max_span_length=3, use_masking=False):
#         super().__init__()
        
#         # Store device info
#         self._device = torch.device('cpu')
#         #self.grid_size = (4, 56)  # Hardcode based on ResNet18 output
#         self.num_patches = self.grid_size[0] * self.grid_size[1]  # 224 patches

#         # Initialize positional embedding
#         self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
#         # Store essential dimensions first
#         self.embed_dim = embed_dim
#         self.num_classes = nb_cls
#         self.mask_ratio = mask_ratio
#         self.max_span_length = max_span_length
#         self.use_masking = use_masking
        
#         # Initialize layers
#         self.layer_norm = nn.LayerNorm(embed_dim)
#         self.patch_embed = ResNet18(embed_dim)
#         self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
#         print(f"Grid size: {self.grid_size}")
        
#         self.num_patches = self.grid_size[0] * self.grid_size[1]
#         print(f"Num patches: {self.num_patches}")
#         self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
#         self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
#         print(f"Positional embeddings shape: {self.pos_embed.shape}")
#         # Should output [batch, 768, 4, 56] for 224x224 input
#         print(f"Confirm ResNet18's output shape:{self.patch_embed(torch.randn(1, 1, 224, 224)).shape}")
#         # Transformer blocks
#         self.blocks = nn.ModuleList([
#             Block(embed_dim, num_heads, self.num_patches, mlp_ratio, 
#                  qkv_bias=True, norm_layer=norm_layer)
#             for _ in range(depth)])
        
#         self.norm = norm_layer(embed_dim)
#         self.head = nn.Linear(embed_dim, nb_cls)
        
#         # Initialize weights
#         self.initialize_weights()

class MaskedAutoencoderViT(nn.Module):
    def __init__(self, nb_cls, img_size=(224, 224), patch_size=(16, 16), embed_dim=768,
                 depth=4, num_heads=6, mlp_ratio=4., norm_layer=nn.LayerNorm,
                 mask_ratio=0.75, max_span_length=3, use_masking=False):
        super().__init__()
        
        # Original configuration preserved
        self._device = torch.device('cpu')
        self.grid_size = (14, 14)
        self.num_patches = 14 * 14
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        self.embed_dim = embed_dim
        self.num_classes = nb_cls
        self.mask_ratio = mask_ratio
        self.max_span_length = max_span_length
        self.use_masking = use_masking
        self.adaptive_pool = nn.AdaptiveAvgPool1d(25) 
        
        # Original layers preserved
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.patch_embed = ResNet18(embed_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))  # +1 for cls_token
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, self.num_patches + 1, mlp_ratio,  # +1 for cls_token
                 qkv_bias=True, norm_layer=norm_layer)
            for _ in range(depth)])

        
        self.norm = norm_layer(embed_dim)
        #self.head = nn.Linear(embed_dim, nb_cls)

        
        
        self.initialize_weights()

    def reset_classifier(self, num_classes):
        
        self.num_classes = num_classes
        self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()
        #print(f"inside reset classifier: {self.embed_dim, num_classes}")
        #self.head = nn.Linear(768,96)
        #self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()
  
    def to(self, device):
        self._device = device
        return super().to(device)
        

    def initialize_weights(self):
        nn.init.normal_(self.cls_token, std=.02)
       
       
        # Generate positional embeddings for the correct grid size
        pos_embed = get_2d_sincos_pos_embed(
            embed_dim=self.embed_dim,
            grid_size=self.grid_size  # (4, 56)
        )
        # Add positional embedding for cls_token (zeros)
        cls_pos_embed = np.zeros((1, self.embed_dim))
        pos_embed = np.concatenate([cls_pos_embed, pos_embed], axis=0)
        self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float().unsqueeze(0))


    
        nn.init.normal_(self.mask_token, std=.02)
        
        # Apply custom initialization
        self.apply(self._init_weights)


    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def generate_span_mask(self, x, mask_ratio, max_span_length):
        N, L, D = x.shape  # batch, length, dim
        mask = torch.ones(N, L, 1).to(x.device)
        span_length = int(L * mask_ratio)
        num_spans = span_length // max_span_length
        for i in range(num_spans):
            idx = torch.randint(L - max_span_length, (1,))
            mask[:,idx:idx + max_span_length,:] = 0
        return mask

    def random_masking(self, x, mask_ratio, max_span_length):
        """
        Perform per-sample random masking by per-sample shuffling.
        Per-sample shuffling is done by argsort random noise.
        x: [N, L, D], sequence
        """
        mask = self.generate_span_mask(x, mask_ratio, max_span_length)
        x_masked = x * mask + (1 - mask) * self.mask_token
        return x_masked



    
    def forward(self, x, seqlen: int =25, mask_ratio=None, max_span_length=None, use_masking=None):
        # print(f"MaskedAutoencoderViT forward Input device: {x.device}")  # Debug
        # print(f"MaskedAutoencoderViT forward Model device: {next(self.parameters()).device}")
        if not any(p.is_cuda for p in self.parameters()):
            # Model is on CPU
            if x.is_cuda:
                x = x.cpu()
        else:
            # Model is on GPU
            if not x.is_cuda:
                x = x.cuda()

        # if next(self.parameters()).is_cuda and not x.is_cuda:
        #     x = x.cuda()
        # elif not next(self.parameters()).is_cuda and x.is_cuda:
        #     x = x.cpu()
        # Original patch embedding
        #print(f"forwarrrrrrrrrd function parameters:{seqlen}")
        #print(f"MaskedAutoEncoderViT forward method Input shape:{x.shape}")
        x = self.patch_embed(x)  # [b, 768, 14, 14]
        #print(f"After patch_embed():{x.shape}")
        x = x.flatten(2).transpose(1, 2)  # [b, 196, 768]
        #print(f"After flatten:{x.shape}")
        x = self.layer_norm(x)
        
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)  # [b, 1, 768]
        x = torch.cat((cls_token, x), dim=1)  # [b, 197, 768]
        #print(f"After adding cls_token: {x.shape}")
        # masking: length -> length * mask_ratio
        if use_masking:
            x = self.random_masking(x, mask_ratio, max_span_length)        
        x = x + self.pos_embed
        #print(f"After pos_embed:{x.shape}")
        
        # Apply transformer blocks
        for blk in self.blocks:
            x = blk(x)
        #print(f"After encoder blocks:{x.shape}")
        x = self.norm(x)
        #print(f"before selectig 25: {x.shape}")
        
        # FIX: Select only the first 'seqlen' tokens for text recognition
        # This maintains patch order while fixing sequence length
        #print(f"sequence length: {seqlen}")
        x = x[:, :seqlen]  # [b, 25, 768]
        #print(f"after selectig 25: {x.shape}")
        # Add validation
        # if x.shape[1] != seqlen:
        #     raise ValueError(f"Sequence length mismatch: {x.shape[1]} != {seqlen}")     
        b, s, e = x.size()

        x = x.reshape(b*s, e)
        #print(f"after reshape: {x.shape}")
        x = self.head(x).view(b, s, self.num_classes)
        return x
        
    

       
   
class ViTSTR(VisionTransformer):
    '''
    ViTSTR is basically a ViT that uses DeiT weights.
    Modified head to support a sequence of characters prediction for STR.
    '''
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.mask_ratio = kwargs.get('mask_ratio', 0.0)#added
        self.use_masking = kwargs.get('use_masking', False)  #added 
    def reset_classifier(self, num_classes):
        self.num_classes = num_classes
        self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()

    def forward_features(self, x):
        # print(f"ViTSTR forward_features Input device: {x.device}")  # Debug
        # print(f"ViTSTR forward_features Model device: {next(self.parameters()).device}")
        mask_ratio = self.mask_ratio if mask_ratio is None else mask_ratio#added
        use_masking = self.use_masking if use_masking is None else use_masking  #added      
        B = x.shape[0]
        
        x = self.patch_embed(x)
        

        cls_tokens = self.cls_token.expand(B, -1, -1)  # stole cls_tokens impl from Phil Wang, thanks
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)
        return x

    def forward(self, x, seqlen: int =25):
        # print(f"ViTstr forward Input device: {x.device}")  # Debug
        # print(f"ViTstr forward Model device: {next(self.parameters()).device}")
        x = self.forward_features(x)
        x = x[:, :seqlen]

        # batch, seqlen, embsize
        b, s, e = x.size()
        x = x.reshape(b*s, e)
        x = self.head(x).view(b, s, self.num_classes)
        return x


def load_pretrained(model, cfg=None, num_classes=1000, in_chans=1, filter_fn=None, strict=True):
    '''
    Loads a pretrained checkpoint
    From an older version of timm
    '''
    if cfg is None:
        cfg = getattr(model, 'default_cfg')
    if cfg is None or 'url' not in cfg or not cfg['url']:
        _logger.warning("Pretrained model URL is invalid, using random initialization.")
        return

    state_dict = model_zoo.load_url(cfg['url'], progress=True, map_location='cpu')
    if "model" in state_dict.keys():
        state_dict = state_dict["model"]

    if filter_fn is not None:
        state_dict = filter_fn(state_dict)

    if in_chans == 1:
        conv1_name = cfg['first_conv']
        _logger.info('Converting first conv (%s) pretrained weights from 3 to 1 channel' % conv1_name)
        key = conv1_name + '.weight'
        if key in state_dict.keys():
            _logger.info('(%s) key found in state_dict' % key)
            conv1_weight = state_dict[conv1_name + '.weight']
        else:
            _logger.info('(%s) key NOT found in state_dict' % key)
            return
        # Some weights are in torch.half, ensure it's float for sum on CPU
        conv1_type = conv1_weight.dtype
        conv1_weight = conv1_weight.float()
        O, I, J, K = conv1_weight.shape
        if I > 3:
            assert conv1_weight.shape[1] % 3 == 0
            # For models with space2depth stems
            conv1_weight = conv1_weight.reshape(O, I // 3, 3, J, K)
            conv1_weight = conv1_weight.sum(dim=2, keepdim=False)
        else:
            conv1_weight = conv1_weight.sum(dim=1, keepdim=True)
        conv1_weight = conv1_weight.to(conv1_type)
        state_dict[conv1_name + '.weight'] = conv1_weight

    classifier_name = cfg['classifier']
    if num_classes == 1000 and cfg['num_classes'] == 1001:
        # special case for imagenet trained models with extra background class in pretrained weights
        classifier_weight = state_dict[classifier_name + '.weight']
        state_dict[classifier_name + '.weight'] = classifier_weight[1:]
        classifier_bias = state_dict[classifier_name + '.bias']
        state_dict[classifier_name + '.bias'] = classifier_bias[1:]
    elif num_classes != cfg['num_classes']:
        # completely discard fully connected for all other differences between pretrained and created model
        del state_dict[classifier_name + '.weight']
        del state_dict[classifier_name + '.bias']
        strict = False

    print("Loading pre-trained vision transformer weights from %s ..." % cfg['url'])
    model.load_state_dict(state_dict, strict=strict)


def _conv_filter(state_dict, patch_size=16):
    """ convert patch embedding weight from manual patchify + linear proj to conv"""
    out_dict = {}
    for k, v in state_dict.items():
        if 'patch_embed.proj.weight' in k:
            v = v.reshape((v.shape[0], 3, patch_size, patch_size))
        out_dict[k] = v
    return out_dict

@register_model
def vitstr_mae_custom(pretrained=False, **kwargs):
    # Default parameters matching the original MAE configuration
    default_kwargs = {
        'img_size': (224, 224),  # Original image size
        'patch_size': (4, 64),  # Original patch size
        'embed_dim': 768,        # Original embedding dimension
        'depth': 4,             # Original number of transformer blocks
        'num_heads': 6,         # Original number of attention heads
        'mlp_ratio': 4,         # Original MLP ratio
        'norm_layer': partial(nn.LayerNorm, eps=1e-6),  # Original norm layer
        'in_chans': 1,          # For text recognition (grayscale)
    }
    
    # Update with any user-provided kwargs
    default_kwargs.update(kwargs)
    
    # Create the model with fixed configuration
    model = MaskedAutoencoderViT(
        nb_cls=default_kwargs.get('num_classes', 96),  # Will be reset later
        img_size=default_kwargs['img_size'],
        patch_size=default_kwargs['patch_size'],
        embed_dim=default_kwargs['embed_dim'],
        depth=default_kwargs['depth'],
        num_heads=default_kwargs['num_heads'],
        mlp_ratio=default_kwargs['mlp_ratio'],
        norm_layer=default_kwargs['norm_layer']
    )
    
    # Handle pretrained weights if needed
    if pretrained:
        # Note: This model likely won't have pretrained weights available
        # You would need to implement loading your own pretrained weights
        pass
    
    return model

In [5]:
def create_vitstr(num_tokens, model=None, checkpoint_path=''):
    vitstr = create_model(
        model,
        pretrained=True,
        num_classes=num_tokens,
        checkpoint_path=checkpoint_path)

    # might need to run to get zero init head for transfer learning
    vitstr.reset_classifier(num_classes=num_tokens)

    return vitstr

In [7]:
device='cuda'
model = create_vitstr(96,'vitstr_mae_custom').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","vitstr_mae_custom")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool1d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
Model :  vitstr_mae_custom
Flops :  34.393393152
macs :  17.196696576
Params :  53.371296


In [8]:
device='cuda'
model = create_vitstr(96,'faster_vit_0_224').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","faster_vit_0_224")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

No pretrained configuration specified for faster_vit_0_224 model. Using a default. Please add a config to the model pretrained_cfg registry or pass explicitly.


---------------------
True
---------------------
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_avgpool() for <class 'torch.nn.modules.pooling.AvgPool2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
Model :  faster_vit_0_224
Flops :  6.891982848
macs :  3.445991424
Params :  30.9416


In [9]:
device='cuda'
model = create_vitstr(96,'faster_vit_2_224').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","faster_vit_2_224")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

No pretrained configuration specified for faster_vit_2_224 model. Using a default. Please add a config to the model pretrained_cfg registry or pass explicitly.


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_avgpool() for <class 'torch.nn.modules.pooling.AvgPool2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
Model :  faster_vit_2_224
Flops :  17.794086912
macs :  8.897043456
Params :  75.229984


In [10]:
device='cuda'
model = create_vitstr(96,'faster_vit_3_224').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","faster_vit_3_224")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

No pretrained configuration specified for faster_vit_3_224 model. Using a default. Please add a config to the model pretrained_cfg registry or pass explicitly.


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_avgpool() for <class 'torch.nn.modules.pooling.AvgPool2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
Model :  faster_vit_3_224
Flops :  37.129740288
macs :  18.564870144
Params :  158.588704


In [9]:
device='cuda'
model = create_vitstr(96,'pcpvt_base_v0').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","pcpvt_base_v0")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
Model :  pcpvt_base_v0
Flops :  12.902907904
macs :  6.451453952
Params :  43.362656


In [10]:
device='cuda'
model = create_vitstr(96,'pcpvt_large_v0').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","pcpvt_large_v0")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
Model :  pcpvt_large_v0
Flops :  19.039558144
macs :  9.519779072
Params :  60.523872


In [11]:
device='cuda'
model = create_vitstr(96,'alt_gvt_small').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","alt_gvt_small")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
Model :  alt_gvt_small
Flops :  5.6303232
macs :  2.8151616
Params :  23.594976


In [12]:
device='cuda'
model = create_vitstr(96,'alt_gvt_base').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","alt_gvt_base")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
Model :  alt_gvt_base
Flops :  16.6944192
macs :  8.3472096
Params :  55.372704


In [13]:
device='cuda'
model = create_vitstr(96,'alt_gvt_large').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","alt_gvt_large")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
Model :  alt_gvt_large
Flops :  29.64914688
macs :  14.82457344
Params :  98.340704


In [14]:
'''
Implementation of ViTSTR based on timm VisionTransformer.

TODO: 
1) distilled deit backbone
2) base deit backbone

Copyright 2021 Rowel Atienza
'''

from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import torch 
import torch.nn as nn
import logging
import torch.utils.model_zoo as model_zoo

from copy import deepcopy
from functools import partial
from timm.models.vision_transformer import VisionTransformer, _cfg
from timm.models.registry import register_model
from timm.models import create_model

_logger = logging.getLogger(__name__)

__all__ = [
    'vitstr_tiny_patch16_224', 
    'vitstr_small_patch16_224', 
    'vitstr_base_patch16_224','pcpvt_small_v0','pcpvt_base_v0','pcpvt_large_v0','alt_gvt_small','alt_gvt_base','alt_gvt_large',
    #'vitstr_tiny_distilled_patch16_224', 
    #'vitstr_small_distilled_patch16_224',
    #'vitstr_base_distilled_patch16_224',
]

def create_vitstr(num_tokens, model=None, checkpoint_path=''):
    vitstr = create_model(
        model,
        pretrained=True,
        num_classes=num_tokens,
        checkpoint_path=checkpoint_path)

    # might need to run to get zero init head for transfer learning
    vitstr.reset_classifier(num_classes=num_tokens)

    return vitstr

class ViTSTR(VisionTransformer):
    '''
    ViTSTR is basically a ViT that uses DeiT weights.
    Modified head to support a sequence of characters prediction for STR.
    '''
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    def reset_classifier(self, num_classes):
        self.num_classes = num_classes
        self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()

    def forward_features(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)  # stole cls_tokens impl from Phil Wang, thanks
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)
        return x

    def forward(self, x, seqlen: int =25):
        x = self.forward_features(x)
        x = x[:, :seqlen]

        # batch, seqlen, embsize
        b, s, e = x.size()
        x = x.reshape(b*s, e)
        x = self.head(x).view(b, s, self.num_classes)
        return x


def load_pretrained(model, cfg=None, num_classes=1000, in_chans=1, filter_fn=None, strict=True):
    '''
    Loads a pretrained checkpoint
    From an older version of timm
    '''
    if cfg is None:
        cfg = getattr(model, 'default_cfg')
    if cfg is None or 'url' not in cfg or not cfg['url']:
        _logger.warning("Pretrained model URL is invalid, using random initialization.")
        return

    state_dict = model_zoo.load_url(cfg['url'], progress=True, map_location='cpu')
    if "model" in state_dict.keys():
        state_dict = state_dict["model"]

    if filter_fn is not None:
        state_dict = filter_fn(state_dict)

    if in_chans == 1:
        conv1_name = cfg['first_conv']
        _logger.info('Converting first conv (%s) pretrained weights from 3 to 1 channel' % conv1_name)
        key = conv1_name + '.weight'
        if key in state_dict.keys():
            _logger.info('(%s) key found in state_dict' % key)
            conv1_weight = state_dict[conv1_name + '.weight']
        else:
            _logger.info('(%s) key NOT found in state_dict' % key)
            return
        # Some weights are in torch.half, ensure it's float for sum on CPU
        conv1_type = conv1_weight.dtype
        conv1_weight = conv1_weight.float()
        O, I, J, K = conv1_weight.shape
        if I > 3:
            assert conv1_weight.shape[1] % 3 == 0
            # For models with space2depth stems
            conv1_weight = conv1_weight.reshape(O, I // 3, 3, J, K)
            conv1_weight = conv1_weight.sum(dim=2, keepdim=False)
        else:
            conv1_weight = conv1_weight.sum(dim=1, keepdim=True)
        conv1_weight = conv1_weight.to(conv1_type)
        state_dict[conv1_name + '.weight'] = conv1_weight

    classifier_name = cfg['classifier']
    if num_classes == 1000 and cfg['num_classes'] == 1001:
        # special case for imagenet trained models with extra background class in pretrained weights
        classifier_weight = state_dict[classifier_name + '.weight']
        state_dict[classifier_name + '.weight'] = classifier_weight[1:]
        classifier_bias = state_dict[classifier_name + '.bias']
        state_dict[classifier_name + '.bias'] = classifier_bias[1:]
    elif num_classes != cfg['num_classes']:
        # completely discard fully connected for all other differences between pretrained and created model
        del state_dict[classifier_name + '.weight']
        del state_dict[classifier_name + '.bias']
        strict = False

    print("Loading pre-trained vision transformer weights from %s ..." % cfg['url'])
    model.load_state_dict(state_dict, strict=strict)


def _conv_filter(state_dict, patch_size=16):
    """ convert patch embedding weight from manual patchify + linear proj to conv"""
    out_dict = {}
    for k, v in state_dict.items():
        if 'patch_embed.proj.weight' in k:
            v = v.reshape((v.shape[0], 3, patch_size, patch_size))
        out_dict[k] = v
    return out_dict

@register_model
def vitstr_tiny_patch16_224(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = ViTSTR(
        patch_size=16, embed_dim=192, depth=12, num_heads=3, mlp_ratio=4, qkv_bias=True, **kwargs)

    model.default_cfg = _cfg(
            #url='https://github.com/roatienza/public/releases/download/v0.1-deit-tiny/deit_tiny_patch16_224-a1311bcf.pth'
            url='https://dl.fbaipublicfiles.com/deit/deit_tiny_patch16_224-a1311bcf.pth'
    )

    if pretrained:
        load_pretrained(
            model, num_classes=model.num_classes, in_chans=kwargs.get('in_chans', 1), filter_fn=_conv_filter)
    return model

@register_model
def vitstr_small_patch16_224(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = ViTSTR(
        patch_size=16, embed_dim=384, depth=12, num_heads=6, mlp_ratio=4, qkv_bias=True, **kwargs)
    model.default_cfg = _cfg(
            #url="https://github.com/roatienza/public/releases/download/v0.1-deit-small/deit_small_patch16_224-cd65a155.pth"
            url="https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth"
    )
    if pretrained:
        load_pretrained(
            model, num_classes=model.num_classes, in_chans=kwargs.get('in_chans', 1), filter_fn=_conv_filter)
    return model

@register_model
def vitstr_base_patch16_224(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = ViTSTR(
        patch_size=16, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4, qkv_bias=True, **kwargs)
    model.default_cfg = _cfg(
            #url='https://github.com/roatienza/public/releases/download/v0.1-deit-base/deit_base_patch16_224-b5f2ef4d.pth'
            url='https://dl.fbaipublicfiles.com/deit/deit_base_patch16_224-b5f2ef4d.pth'
    )
    if pretrained:
        load_pretrained(
            model, num_classes=model.num_classes, in_chans=kwargs.get('in_chans', 1), filter_fn=_conv_filter)
    return model

# below is work in progress
@register_model
def vitstr_tiny_distilled_patch16_224(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    #kwargs['distilled'] = True
    model = ViTSTR(
        patch_size=16, embed_dim=192, depth=12, num_heads=3, mlp_ratio=4, qkv_bias=True, **kwargs)
    model.default_cfg = _cfg(
            url='https://dl.fbaipublicfiles.com/deit/deit_tiny_distilled_patch16_224-b40b3cf7.pth'
    )

    if pretrained:
        load_pretrained(
            model, num_classes=model.num_classes, in_chans=kwargs.get('in_chans', 1), filter_fn=_conv_filter)
    return model


@register_model
def vitstr_small_distilled_patch16_224(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    kwargs['distilled'] = True
    model = ViTSTR(
        patch_size=16, embed_dim=384, depth=12, num_heads=6, mlp_ratio=4, qkv_bias=True, **kwargs)
    model.default_cfg = _cfg(
            url="https://dl.fbaipublicfiles.com/deit/deit_small_distilled_patch16_224-649709d9.pth"
    )
    if pretrained:
        load_pretrained(
            model, num_classes=model.num_classes, in_chans=kwargs.get('in_chans', 1), filter_fn=_conv_filter)
    return model



import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial

from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from timm.models.registry import register_model
from timm.models.vision_transformer import _cfg
import math

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1)
        self.dwconv = DWConv(hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1)
        self.drop = nn.Dropout(drop)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x):
        x = self.fc1(x)
        x = self.dwconv(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x




class LKA(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv0 = nn.Conv2d(dim, dim, 5, padding=2, groups=dim)
        self.conv_spatial = nn.Conv2d(dim, dim, 7, stride=1, padding=9, groups=dim, dilation=3)
        self.conv1 = nn.Conv2d(dim, dim, 1)


    def forward(self, x):
        u = x.clone()        
        attn = self.conv0(x)
        attn = self.conv_spatial(attn)
        attn = self.conv1(attn)

        return u * attn


class Attention(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.proj_1 = nn.Conv2d(d_model, d_model, 1)
        self.activation = nn.GELU()
        self.spatial_gating_unit = LKA(d_model)
        self.proj_2 = nn.Conv2d(d_model, d_model, 1)

    def forward(self, x):
        shorcut = x.clone()
        x = self.proj_1(x)
        x = self.activation(x)
        x = self.spatial_gating_unit(x)
        x = self.proj_2(x)
        x = x + shorcut
        return x


class Block(nn.Module):
    def __init__(self, dim, mlp_ratio=4., drop=0.,drop_path=0., act_layer=nn.GELU):
        super().__init__()
        self.norm1 = nn.BatchNorm2d(dim)
        self.attn = Attention(dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

        self.norm2 = nn.BatchNorm2d(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)
        layer_scale_init_value = 1e-2            
        self.layer_scale_1 = nn.Parameter(
            layer_scale_init_value * torch.ones((dim)), requires_grad=True)
        self.layer_scale_2 = nn.Parameter(
            layer_scale_init_value * torch.ones((dim)), requires_grad=True)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x):
        x = x + self.drop_path(self.layer_scale_1.unsqueeze(-1).unsqueeze(-1) * self.attn(self.norm1(x)))
        x = x + self.drop_path(self.layer_scale_2.unsqueeze(-1).unsqueeze(-1) * self.mlp(self.norm2(x)))
        return x


class OverlapPatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """

    def __init__(self, img_size=224, patch_size=7, stride=4, in_chans=3, embed_dim=768):
        super().__init__()
        patch_size = to_2tuple(patch_size)
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride,
                              padding=(patch_size[0] // 2, patch_size[1] // 2))
        self.norm = nn.BatchNorm2d(embed_dim)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = self.norm(x)        
        return x, H, W


class VAN(nn.Module):
    def __init__(self, img_size=224, in_chans=3, num_classes=1000, embed_dims=[64, 128, 256, 512],
                mlp_ratios=[4, 4, 4, 4], drop_rate=0., drop_path_rate=0., norm_layer=nn.LayerNorm,
                 depths=[3, 4, 6, 3], num_stages=4, flag=False):
        super().__init__()
        if flag == False:
            self.num_classes = num_classes
        self.depths = depths
        self.num_stages = num_stages

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule
        cur = 0

        for i in range(num_stages):
            patch_embed = OverlapPatchEmbed(img_size=img_size if i == 0 else img_size // (2 ** (i + 1)),
                                            patch_size=7 if i == 0 else 3,
                                            stride=4 if i == 0 else 2,
                                            in_chans=in_chans if i == 0 else embed_dims[i - 1],
                                            embed_dim=embed_dims[i])

            block = nn.ModuleList([Block(
                dim=embed_dims[i], mlp_ratio=mlp_ratios[i], drop=drop_rate, drop_path=dpr[cur + j])
                for j in range(depths[i])])
            norm = norm_layer(embed_dims[i])
            cur += depths[i]

            setattr(self, f"patch_embed{i + 1}", patch_embed)
            setattr(self, f"block{i + 1}", block)
            setattr(self, f"norm{i + 1}", norm)

        # classification head
        self.embed_dim = embed_dims[-1]  # Added line to set embed_dim
        self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def freeze_patch_emb(self):
        self.patch_embed1.requires_grad = False

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'pos_embed1', 'pos_embed2', 'pos_embed3', 'pos_embed4', 'cls_token'}  # has pos_embed may be better

    def get_classifier(self):
        return self.head

    def reset_classifier(self, num_classes, global_pool=''):
        self.num_classes = num_classes
        self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()

    def forward(self, x, seqlen=None):
        # x=x.repeat(1, 3, 1, 1)
        for i in range(self.num_stages):
            patch_embed = getattr(self, f"patch_embed{i + 1}")
            block = getattr(self, f"block{i + 1}")
            norm = getattr(self, f"norm{i + 1}")
            x, H, W = patch_embed(x)
            for blk in block:
                x = blk(x)
            x = x.flatten(2).transpose(1, 2)
            x = norm(x)
            if i != self.num_stages - 1:
                #x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
                x = x.reshape(x.size(0), H, W, -1).permute(0, 3, 1, 2).contiguous()


        # x = x.mean(dim=1)
        if seqlen is not None:
            # Handle the 'seqlen' argument if provided
            # You might want to perform some operations based on 'seqlen'
            pass
        x = x[:, :27]
        x = self.head(x)

        return x

class DWConv(nn.Module):
    def __init__(self, dim=768):
        super(DWConv, self).__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, bias=True, groups=dim)

    def forward(self, x):
        x = self.dwconv(x)
        return x


def _conv_filter(state_dict, patch_size=16):
    """ convert patch embedding weight from manual patchify + linear proj to conv"""
    out_dict = {}
    for k, v in state_dict.items():
        if 'patch_embed.proj.weight' in k:
            v = v.reshape((v.shape[0], 3, patch_size, patch_size))
        out_dict[k] = v

    return out_dict


# model_urls = {
#     "van_b0": "https://huggingface.co/Visual-Attention-Network/VAN-Tiny-original/resolve/main/van_tiny_754.pth.tar",
#     "van_b1": "https://huggingface.co/Visual-Attention-Network/VAN-Small-original/resolve/main/van_small_811.pth.tar",
#     "van_b2": "https://huggingface.co/Visual-Attention-Network/VAN-Base-original/resolve/main/van_base_828.pth.tar",
#     "van_b3": "https://huggingface.co/Visual-Attention-Network/VAN-Large-original/resolve/main/van_large_839.pth.tar",
# }
model_urls = {
    "van_b0": "",
    "van_b1": "",
    "van_b2": "",
    "van_b3": "",
    "van_b4": "",
    "van_b5": "",
    "van_b6": "",
}


def load_model_weights(model, arch, kwargs):
    # return model

    url = model_urls[arch]
    if not url.strip():
        print("No pretrained model given in url field ... ")
        return model
    checkpoint = torch.hub.load_state_dict_from_url(
        url=url, map_location="cpu", check_hash=True
    )
    strict = True
    if "num_classes" in kwargs and kwargs["num_classes"] != 1000:
        strict = False
        del checkpoint["state_dict"]["head.weight"]
        del checkpoint["state_dict"]["head.bias"]
    model.load_state_dict(checkpoint["state_dict"], strict=strict)
    return model


@register_model
def van_b0(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = VAN(
        embed_dims=[32, 64, 160, 256], mlp_ratios=[8, 8, 4, 4],
        norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 3, 5, 2],
        **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        model = load_model_weights(model, "van_b0", kwargs)
    return model


@register_model
def van_b1(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = VAN(
        embed_dims=[64, 128, 320, 512], mlp_ratios=[8, 8, 4, 4],
        norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[2, 2, 4, 2],
        **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        model = load_model_weights(model, "van_b1", kwargs)
    return model

@register_model
def van_b2(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = VAN(
        embed_dims=[64, 128, 320, 512], mlp_ratios=[8, 8, 4, 4],
        norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 3, 12, 3],
        **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        model = load_model_weights(model, "van_b2", kwargs)
    return model

@register_model
def van_b3(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = VAN(
        embed_dims=[64, 128, 320, 512], mlp_ratios=[8, 8, 4, 4],
        norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 5, 27, 3],
        **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        model = load_model_weights(model, "van_b3", kwargs)
    return model

@register_model
def van_b4(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = VAN(
        embed_dims=[64, 128, 320, 512], mlp_ratios=[8, 8, 4, 4],
        norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 6, 40, 3],
        **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        model = load_model_weights(model, "van_b4", kwargs)
    return model


@register_model
def van_b5(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = VAN(
        embed_dims=[96, 192, 480, 768], mlp_ratios=[8, 8, 4, 4],
        norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 3, 24, 3],
        **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        model = load_model_weights(model, "van_b5", kwargs)
    return model


@register_model
def van_b6(pretrained=False, **kwargs):
    kwargs['in_chans'] = 1
    model = VAN(
        embed_dims=[96, 192, 384, 768], mlp_ratios=[8, 8, 4, 4],
        norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[6,6,90,6],
        **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        model = load_model_weights(model, "van_b6", kwargs)
    return model



In [15]:
device='cuda'
model = create_vitstr(96,'van_b0').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","van_b0")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

No pretrained model given in url field ... 
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
Model :  van_b0
Flops :  1.722840576
macs :  0.861420288
Params :  3.867136


In [16]:
device='cuda'
model = create_vitstr(96,'van_b1').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","van_b1")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)


No pretrained model given in url field ... 
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
Model :  van_b1
Flops :  4.973490176
macs :  2.486745088
Params :  13.3864


In [17]:
device='cuda'
model = create_vitstr(96,'van_b2').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","van_b2")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

No pretrained model given in url field ... 
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
Model :  van_b2
Flops :  9.98175744
macs :  4.99087872
Params :  26.096544


In [18]:
device='cuda'
model = create_vitstr(96,'van_b3').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","van_b3")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

No pretrained model given in url field ... 
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
Model :  van_b3
Flops :  17.865059328
macs :  8.932529664
Params :  44.273568


In [19]:
device='cuda'
model = create_vitstr(96,'van_b4').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","van_b4")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

No pretrained model given in url field ... 
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
Model :  van_b4
Flops :  24.316639232
macs :  12.158319616
Params :  59.78256


In [20]:
device='cuda'
model = create_vitstr(96,'van_b5').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","van_b5")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

No pretrained model given in url field ... 
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
Model :  van_b5
Flops :  34.270626816
macs :  17.135313408
Params :  89.233728


In [21]:
device='cuda'
model = create_vitstr(96,'van_b6').to(device)
model.eval()
with torch.no_grad():
    input = torch.randn(1, 1, 224, 224).to(device)
    macs, params = profile(model, inputs=(input, ))
macs , params
print("Model : ","van_b6")
print("Flops : ",2*macs/1e9)
print("macs : ",macs/1e9)
print("Params : ",params/1e6,)

No pretrained model given in url field ... 
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
Model :  van_b6
Flops :  77.528008704
macs :  38.764004352
Params :  199.04304
